In [5]:
from osgeo import ogr

from usgs_api.naip import NAIPSceneSearchRequest
from usgs_api.data_types import SpatialFilterMbr, Coordinate, SceneFilter, GeoJson

p = ogr.Geometry(ogr.wkbPoint)
p.AddPoint_2D(-104.585, 39.055)
poly = p.Buffer(0.00001, 1)
lon1, lon2, lat1, lat2 = poly.GetEnvelope()
ll_lat = min(lat1, lat2)
ll_lon = min(lon1, lon2)
ur_lat = max(lat1, lat2)
ur_lon = max(lon1, lon2)

spatial_filter = SpatialFilterMbr(lower_left=Coordinate(ll_lat, ll_lon), upper_right=Coordinate(ur_lat, ur_lon))
scene_filter = SceneFilter(spatial_filter=spatial_filter)

results = []
total_hits = 1
while len(results) < total_hits:
    request = NAIPSceneSearchRequest(scene_filter=scene_filter, starting_number=len(results) + 1, max_results=1000)
    response = request.make_request()
    total_hits = response.data['totalHits']
    results.extend(response.data['results'])
    if len(results) < total_hits:
        print(f'Got {len(results):d} of {total_hits:d} results. Requesting more...')

print(len(results))


{'datasetName': 'naip', 'startingNumber': 1, 'maxResults': 1000, 'sceneFilter': {'spatialFilter': {'filterType': 'mbr', 'lowerLeft': {'latitude': 39.05499, 'longitude': -104.58501}, 'upperRight': {'latitude': 39.05501, 'longitude': -104.58498999999999}}}}
6


In [6]:
from usgs_api.naip import NAIPSceneSearchResult, naip_results_to_shapefile

results_naip = [NAIPSceneSearchResult.from_dict(d) for d in results]
naip_results_to_shapefile(results_naip)

In [7]:
results_naip[0].spatial_coverage_json

{'type': 'Polygon',
 'coordinates': [[[-104.6276638, 38.9981749],
   [-104.5601805, 38.9979388],
   [-104.5597694, 39.0642805],
   [-104.6273166, 39.0645138],
   [-104.6276638, 38.9981749]]]}